# 02 - CBF Evaluation

This notebook evaluates TF-IDF, Word2Vec, and Sentence Embedding CBF models side by side.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from cbf import load_all_cbf, load_fit_times

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
base_dir = Path('.').resolve()
data_path = base_dir / 'books_processed.csv'
if not data_path.exists():
    raise FileNotFoundError('books_processed.csv not found. Run features.py first.')

t0 = time.perf_counter()
models = load_all_cbf(str(data_path))
load_elapsed = time.perf_counter() - t0
print(f'Loaded all models in {load_elapsed:.2f} seconds')

df = pd.read_csv(data_path)
for col in ['title', 'genre', 'subjects']:
    if col not in df.columns:
        df[col] = ''

fit_times = load_fit_times(str(base_dir / 'model_fit_times.json'))
fit_times

In [ ]:
def resolve_title(query_title, df_titles):
    titles = pd.Series(df_titles).fillna('').astype(str)
    if (titles.str.lower() == query_title.lower()).any():
        return titles[titles.str.lower() == query_title.lower()].iloc[0]
    contains = titles[titles.str.lower().str.contains(query_title.lower(), regex=False)]
    if len(contains):
        return contains.iloc[0]
    return None

def split_subjects(subjects):
    if not isinstance(subjects, str):
        return set()
    return {x.strip().lower() for x in subjects.split(';') if x.strip()}

def jaccard(a, b):
    if not a and not b:
        return 0.0
    union = a.union(b)
    if not union:
        return 0.0
    return len(a.intersection(b)) / len(union)

## Section 1 - Recommendation Spot-Check

In [ ]:
test_books = [
    'Project Hail Mary',
    'Atomic Habits',
    'The Psychology of Money',
    'Lessons in Chemistry',
    'Chip War',
    'Think Again',
    'Empire of Pain',
    'The Midnight Library',
    'Outlive',
    'The Dawn of Everything'
]

spotcheck_tables = []
for title in test_books:
    resolved = resolve_title(title, df['title'])
    if not resolved:
        continue

    block = pd.DataFrame({'query_title': [resolved] * 5, 'rank': [1, 2, 3, 4, 5]})
    for model_name, model in models.items():
        recs = model.get_recommendations(resolved, top_n=5)
        block[f'{model_name}_title'] = recs['title'].values
        block[f'{model_name}_genre'] = recs['genre'].values
        block[f'{model_name}_score'] = recs['similarity_score'].round(4).values
    spotcheck_tables.append(block)

if spotcheck_tables:
    pd.concat(spotcheck_tables, ignore_index=True).head(50)
else:
    print('None of the test books were found in the dataset.')

## Section 2 - Genre Precision@K

In [ ]:
K = 5
genre_precision_by_model = {}
genre_precision_breakdown = {}

for model_name, model in models.items():
    per_book = []
    rows = []
    for _, row in df.iterrows():
        title = str(row['title'])
        genre = str(row['genre'])
        if not title:
            continue
        try:
            recs = model.get_recommendations(title, top_n=K)
        except Exception:
            continue

        precision = (recs['genre'].astype(str) == genre).mean()
        per_book.append(precision)
        rows.append({'genre': genre, 'precision': precision})

    genre_precision_by_model[model_name] = float(np.mean(per_book)) if per_book else 0.0
    genre_precision_breakdown[model_name] = pd.DataFrame(rows).groupby('genre')['precision'].mean().reset_index() if rows else pd.DataFrame(columns=['genre', 'precision'])

genre_precision_by_model

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=list(genre_precision_by_model.keys()), y=list(genre_precision_by_model.values()))
plt.title('Mean Precision@5 by Model')
plt.ylabel('Precision@5')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for model_name, gdf in genre_precision_breakdown.items():
    if gdf.empty:
        continue
    tmp = gdf.copy()
    tmp['model'] = model_name
    rows.append(tmp)

if rows:
    breakdown_df = pd.concat(rows, ignore_index=True)
    top_genres = breakdown_df.groupby('genre')['precision'].mean().sort_values(ascending=False).head(12).index
    plot_df = breakdown_df[breakdown_df['genre'].isin(top_genres)]

    plt.figure(figsize=(14, 6))
    sns.barplot(data=plot_df, x='genre', y='precision', hue='model')
    plt.xticks(rotation=50, ha='right')
    plt.title('Per-Genre Precision@5 by Model')
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print('No genre breakdown data available.')

## Section 3 - Tag Overlap (Jaccard@K)

In [ ]:
jaccard_scores = {}

for model_name, model in models.items():
    per_pair = []
    for _, row in df.iterrows():
        title = str(row['title'])
        if not title:
            continue
        query_tags = split_subjects(row['subjects'])
        try:
            recs = model.get_recommendations(title, top_n=5)
        except Exception:
            continue

        for _, rec in recs.iterrows():
            rec_tags = split_subjects(rec['subjects'])
            per_pair.append(jaccard(query_tags, rec_tags))

    jaccard_scores[model_name] = float(np.mean(per_pair)) if per_pair else 0.0

jaccard_scores

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=list(jaccard_scores.keys()), y=list(jaccard_scores.values()))
plt.title('Mean Jaccard@5 by Model')
plt.ylabel('Jaccard@5')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Section 4 - Similarity Score Distribution

In [ ]:
similarity_distributions = {}
stats_rows = []

for model_name, model in models.items():
    scores = []
    for title in df['title'].fillna('').astype(str):
        if not title:
            continue
        try:
            recs = model.get_recommendations(title, top_n=5)
        except Exception:
            continue
        scores.extend(recs['similarity_score'].astype(float).tolist())

    scores = np.array(scores, dtype=float)
    similarity_distributions[model_name] = scores

    if len(scores):
        stats_rows.append({
            'model': model_name,
            'mean': float(scores.mean()),
            'median': float(np.median(scores)),
            'std': float(scores.std()),
            'pct_above_0_3': float((scores > 0.3).mean() * 100),
        })

score_stats_df = pd.DataFrame(stats_rows)
score_stats_df

In [ ]:
plt.figure(figsize=(12, 6))
for model_name, scores in similarity_distributions.items():
    if len(scores) == 0:
        continue
    sns.kdeplot(scores, fill=True, alpha=0.25, label=model_name)

plt.title('Similarity Score Distributions (Top-5 Pairs)')
plt.xlabel('Cosine Similarity')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

## Section 5 - Cold Item Analysis

In [ ]:
cold_items = {}
cold_counts = {}

for model_name, model in models.items():
    records = []
    for _, row in df.iterrows():
        title = str(row['title'])
        genre = str(row['genre'])
        if not title:
            continue

        try:
            recs = model.get_recommendations(title, top_n=5)
        except Exception:
            continue

        best_score = float(recs['similarity_score'].max()) if len(recs) else 0.0
        if best_score < 0.1:
            records.append({'title': title, 'genre': genre, 'best_score': best_score})

    cold_df = pd.DataFrame(records)
    cold_items[model_name] = cold_df
    cold_counts[model_name] = int(len(cold_df))

cold_counts

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=list(cold_counts.keys()), y=list(cold_counts.values()))
plt.title('Cold Item Count by Model (Best Score < 0.1)')
plt.ylabel('Cold Item Count')
plt.tight_layout()
plt.show()

In [ ]:
for model_name, cold_df in cold_items.items():
    print(f'\nModel: {model_name} | Cold items: {len(cold_df)}')
    if len(cold_df):
        display(cold_df.sort_values('best_score').head(20))

## Section 6 - Overall Model Comparison

In [ ]:
summary_rows = []
for model_name in models.keys():
    scores = similarity_distributions.get(model_name, np.array([]))
    summary_rows.append({
        'model': model_name,
        'precision_at_5': genre_precision_by_model.get(model_name, 0.0),
        'jaccard_at_5': jaccard_scores.get(model_name, 0.0),
        'mean_similarity': float(scores.mean()) if len(scores) else 0.0,
        'cold_item_count': cold_counts.get(model_name, 0),
        'fit_time_sec': fit_times.get(model_name, np.nan),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
radar_df = summary_df.copy()
if len(radar_df):
    metrics = ['precision_at_5', 'jaccard_at_5', 'mean_similarity', 'cold_item_count']

    # Invert cold item count because lower is better
    radar_df['cold_item_count'] = radar_df['cold_item_count'].max() - radar_df['cold_item_count']

    for m in metrics:
        col = radar_df[m].astype(float)
        denom = (col.max() - col.min())
        radar_df[m] = 0.0 if denom == 0 else (col - col.min()) / denom

    labels = metrics
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    for _, row in radar_df.iterrows():
        values = row[labels].tolist()
        values += values[:1]
        ax.plot(angles, values, linewidth=2, label=row['model'])
        ax.fill(angles, values, alpha=0.15)

    ax.set_thetagrids(np.degrees(angles[:-1]), labels)
    ax.set_title('Model Comparison Radar Chart (Normalized)')
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()
else:
    print('Summary metrics are not available.')

### Interpretation

Use the summary table and radar chart above to identify the best-performing model for your dataset. In general:

- TF-IDF tends to be fast and strong when subjects/genres are clean and consistent.
- Word2Vec can capture semantic proximity between related words but may underperform on sparse metadata.
- Sentence Embedding usually offers richer semantic understanding, often improving recommendation quality at the cost of heavier compute.

The final choice should balance quality metrics (Precision@K, Jaccard@K, mean similarity) with runtime constraints and cold-item behavior.